# 🚶 People Entry Counter V2 — Door Entrance Monitoring

## Approach: YOLOv8 Detection + BoT-SORT Tracking (Kalman Filter + Hungarian Algorithm) + Virtual Line Crossing

### Improvements over V1
- **Polygon ROI** instead of rectangle — masks out building interior visible through door
- **Min bounding box area filter** — rejects small/partial detections seen through door gap
- **Custom BoT-SORT config** — tuned `track_buffer=60` for better re-ID after door occlusion
- **Foot-level centroid** — uses bottom-center of bbox for stable position tracking

### Pipeline
```
Frame → YOLOv8 Person Detection → Filter (confidence, area, polygon ROI)
      → BoT-SORT Tracking (Kalman Filter prediction + Hungarian Algorithm assignment)
      → Centroid Line-Crossing Detection → Direction Classification → Count
```

### How BoT-SORT uses Kalman Filter & Hungarian Algorithm
1. **Kalman Filter**: Maintains a state vector `[cx, cy, aspect_ratio, height, vx, vy, va, vh]` for each track. Predicts the next position even when detection is temporarily lost (e.g., person occluded by door frame).
2. **Hungarian Algorithm**: Solves the optimal assignment problem — matches current detections to existing tracks by minimizing a cost matrix (IoU distance + appearance similarity). This ensures stable ID assignment even in crowded scenes.

### Direction Classification
- Virtual vertical counting line at the door frame
- **Right → Left** crossing = **ENTERING** (outside → inside)
- **Left → Right** crossing = **EXITING** (inside → outside)
- Verified by trajectory history (last 10 centroid positions)

## 1. Setup & Installation

In [ ]:
# Install dependencies (uncomment if needed)
# !pip install ultralytics opencv-python numpy supervision lapx

In [ ]:
import cv2
import numpy as np
from collections import defaultdict, deque
from ultralytics import YOLO
from pathlib import Path
import time

print("All imports successful!")

## 2. Configuration

All tunable parameters are defined here. Key improvements:
- **Polygon ROI** — defines a precise zone around the door, excluding building interior
- **MIN_BBOX_AREA** — filters out small detections visible through the door opening
- **Custom tracker config** — BoT-SORT with tuned parameters for door occlusion

In [ ]:
# =====================================================================
# CONFIGURATION — Adjust these parameters to match your video
# =====================================================================

# --- Paths ---
VIDEO_INPUT = "entrance.mov"          # Input video file
VIDEO_OUTPUT = "entrance_result.mp4"  # Output annotated video

# --- YOLO Model ---
MODEL_NAME = "yolov8m.pt"    # Model: yolov8n/s/m/l/x (.pt)
CONFIDENCE_THRESHOLD = 0.4   # Min detection confidence (raised from 0.35)
PERSON_CLASS_ID = 0          # COCO class ID for 'person'

# --- Detection Filters ---
MIN_BBOX_AREA = 15000        # Min bounding box area (pixels²)
                              # Filters out small detections seen through door gap
                              # Typical person bbox: ~200x400 = 80,000 px²
                              # Small/partial person through door: ~5,000-10,000 px²

# --- Counting Line (virtual tripwire) ---
# Vertical line near the door frame.
LINE_X = 620               # X position of the vertical counting line
LINE_Y_START = 150         # Top of the line (y)
LINE_Y_END = 750           # Bottom of the line (y)

# --- Polygon ROI ---
# Instead of a rectangle, we use a polygon to precisely define the
# counting zone around the door. This excludes the building interior
# visible through the door opening, reducing false detections.
#
# Points are defined clockwise: (x, y)
# Adjust these to match your camera view.
ROI_POLYGON = np.array([
    [350, 100],    # Top-left (just left of door frame)
    [1100, 100],   # Top-right (well past door, covers outdoor area)
    [1100, 850],   # Bottom-right
    [350, 850],    # Bottom-left (just left of door frame)
], dtype=np.int32)

# Also define an EXCLUSION zone for the building interior visible through door
# This masks out the area inside the building that's visible through the opening
EXCLUDE_POLYGON = np.array([
    [250, 120],    # Top-left of interior visible area
    [550, 120],    # Top-right of interior visible area  
    [550, 650],    # Bottom-right
    [250, 650],    # Bottom-left
], dtype=np.int32)
USE_EXCLUSION_ZONE = False  # Set True to exclude interior detections

# --- Tracking ---
TRACKER_CONFIG = "custom_botsort.yaml"  # Custom BoT-SORT config
TRACK_HISTORY_LENGTH = 50   # Number of past centroid positions to keep
MIN_TRACK_HITS = 5          # Min frames a track must exist before counting
CROSSING_MARGIN = 40        # Pixel margin to confirm a crossing (hysteresis)

# --- Performance ---
FRAME_SKIP = 3        # Process 1 frame every N frames

# --- Visualization ---
SHOW_PREVIEW = False  # Set True to show live preview (requires display)
TRAIL_LENGTH = 30     # Length of trajectory trail drawn on video

print(f"Configuration loaded (V2 — Enhanced).")
print(f"  Input:  {VIDEO_INPUT}")
print(f"  Output: {VIDEO_OUTPUT}")
print(f"  Model:  {MODEL_NAME}")
print(f"  Min bbox area: {MIN_BBOX_AREA} px²")
print(f"  Counting line at x={LINE_X}, y=[{LINE_Y_START}, {LINE_Y_END}]")
print(f"  ROI Polygon: {len(ROI_POLYGON)} points")
print(f"  Tracker: {TRACKER_CONFIG}")

## 3. Core Logic: People Counter V2

Key improvements:
- **Polygon ROI** via `cv2.pointPolygonTest()` instead of simple rectangle
- **Bounding box area filter** to reject small/partial detections
- **Custom BoT-SORT** with `track_buffer=60` for better re-ID after door occlusion

### How the algorithms work together:
```
┌─────────────────────────────────────────────────────────┐
│  YOLOv8 Detection                                       │
│  ├── Detects all persons in frame                       │
│  └── Outputs: bounding boxes + confidence scores        │
│                                                         │
│  BoT-SORT Tracker                                       │
│  ├── Kalman Filter: Predicts next position of each      │
│  │   track based on velocity model                      │
│  ├── Hungarian Algorithm: Optimally assigns detections  │
│  │   to tracks (minimizes IoU + appearance cost)        │
│  └── Outputs: bounding boxes + persistent track IDs     │
│                                                         │
│  Post-Processing                                        │
│  ├── Filter by bbox area (reject small detections)      │
│  ├── Filter by polygon ROI (reject interior detections) │
│  ├── Track centroid history                              │
│  └── Check line crossing + classify direction            │
└─────────────────────────────────────────────────────────┘
```

In [ ]:
class PeopleCounterV2:
    """
    People Entry Counter V2 using YOLOv8 + BoT-SORT + Line Crossing.
    
    Improvements over V1:
        - Polygon ROI (cv2.pointPolygonTest) instead of rectangle
        - Min bounding box area filter
        - Custom BoT-SORT config with tuned parameters
        - Exclusion zone for building interior
    
    Tracking internals (BoT-SORT):
        - Kalman Filter: state = [cx, cy, ar, h, vx, vy, va, vh]
          Predicts position when detection lost (door occlusion)
        - Hungarian Algorithm: optimal detection-to-track assignment
          Cost = IoU distance + appearance distance
    """
    
    def __init__(self, model_name, confidence, line_x, line_y_start, line_y_end,
                 roi_polygon, crossing_margin, track_history_len, min_track_hits,
                 min_bbox_area, tracker_config, exclude_polygon=None, frame_skip=1):
        # Load YOLO model
        self.model = YOLO(model_name)
        self.confidence = confidence
        self.person_class_id = PERSON_CLASS_ID
        
        # Counting line
        self.line_x = line_x
        self.line_y_start = line_y_start
        self.line_y_end = line_y_end
        
        # Polygon ROI (for cv2.pointPolygonTest)
        self.roi_polygon = roi_polygon
        self.exclude_polygon = exclude_polygon
        
        # Detection filter
        self.min_bbox_area = min_bbox_area
        self.tracker_config = tracker_config
        
        # Tracking parameters
        self.crossing_margin = crossing_margin
        self.min_track_hits = min_track_hits
        
        # Per-track state
        self.track_history = defaultdict(lambda: deque(maxlen=track_history_len))
        self.track_side = {}       # Last known side: 'left' or 'right'
        self.counted_ids = set()   # Already counted IDs
        self.track_frames = defaultdict(int)
        
        # Counters
        self.enter_count = 0
        self.exit_count = 0
        self.events = []
        
        # Stats
        self.filtered_by_area = 0
        self.filtered_by_roi = 0
        self.frame_skip = frame_skip
    
    def _get_centroid(self, bbox):
        """Bottom-center centroid (foot level) — more stable for walking people."""
        x1, y1, x2, y2 = bbox
        cx = (x1 + x2) / 2
        cy = y2  # Bottom of bbox
        return cx, cy
    
    def _bbox_area(self, bbox):
        """Calculate bounding box area."""
        x1, y1, x2, y2 = bbox
        return (x2 - x1) * (y2 - y1)
    
    def _is_in_polygon_roi(self, cx, cy):
        """Check if point is inside the polygon ROI and outside exclusion zone.
        
        Uses cv2.pointPolygonTest which returns:
            > 0: inside polygon
            = 0: on edge
            < 0: outside polygon
        """
        # Must be inside ROI polygon
        inside_roi = cv2.pointPolygonTest(
            self.roi_polygon, (float(cx), float(cy)), False
        ) >= 0
        
        if not inside_roi:
            return False
        
        # Must NOT be inside exclusion zone (building interior)
        if self.exclude_polygon is not None:
            inside_exclude = cv2.pointPolygonTest(
                self.exclude_polygon, (float(cx), float(cy)), False
            ) >= 0
            if inside_exclude:
                return False
        
        return True
    
    def _determine_side(self, cx):
        """Determine side of counting line with hysteresis margin."""
        if cx < self.line_x - self.crossing_margin:
            return 'left'
        elif cx > self.line_x + self.crossing_margin:
            return 'right'
        return None
    
    def _check_crossing(self, track_id, cx, cy, frame_no, timestamp):
        """Check if a track crossed the counting line.
        
        Direction classification:
            right → left (decreasing x) = ENTER (into building)
            left → right (increasing x) = EXIT (out of building)
        
        Verified by trajectory analysis (last 10 positions).
        """
        if track_id in self.counted_ids:
            return None
        
        if not self._is_in_polygon_roi(cx, cy):
            self.filtered_by_roi += 1
            return None
        
        if self.track_frames[track_id] < self.min_track_hits:
            return None
        
        current_side = self._determine_side(cx)
        if current_side is None:
            return None
        
        prev_side = self.track_side.get(track_id)
        self.track_side[track_id] = current_side
        
        if prev_side is None or prev_side == current_side:
            return None
        
        history = self.track_history[track_id]
        if len(history) < 3:
            return None
        
        # Trajectory confirmation: verify direction from recent history
        recent = list(history)[-10:]
        x_positions = [p[0] for p in recent]
        avg_dx = x_positions[-1] - x_positions[0]
        
        if prev_side == 'right' and current_side == 'left' and avg_dx < 0:
            self.enter_count += 1
            self.counted_ids.add(track_id)
            direction = 'ENTER'
            self.events.append({
                'frame': frame_no, 'track_id': track_id,
                'direction': direction, 'timestamp': timestamp
            })
            return direction
        
        elif prev_side == 'left' and current_side == 'right' and avg_dx > 0:
            self.exit_count += 1
            self.counted_ids.add(track_id)
            direction = 'EXIT'
            self.events.append({
                'frame': frame_no, 'track_id': track_id,
                'direction': direction, 'timestamp': timestamp
            })
            return direction
        
        return None
    
    def _draw_overlay(self, frame, detections, filtered_count, frame_no, fps):
        """Draw all visual overlays."""
        h, w = frame.shape[:2]
        
        # --- Draw Polygon ROI (green, semi-transparent) ---
        roi_overlay = frame.copy()
        cv2.polylines(roi_overlay, [self.roi_polygon], True, (0, 255, 0), 2, cv2.LINE_AA)
        # Dim outside ROI
        mask = np.zeros((h, w), dtype=np.uint8)
        cv2.fillPoly(mask, [self.roi_polygon], 255)
        if self.exclude_polygon is not None and USE_EXCLUSION_ZONE:
            cv2.fillPoly(mask, [self.exclude_polygon], 0)
        frame_dimmed = frame.copy()
        frame_dimmed[mask == 0] = (frame_dimmed[mask == 0] * 0.5).astype(np.uint8)
        frame = frame_dimmed
        cv2.polylines(frame, [self.roi_polygon], True, (0, 255, 0), 2, cv2.LINE_AA)
        
        # --- Draw exclusion zone (red, if enabled) ---
        if self.exclude_polygon is not None and USE_EXCLUSION_ZONE:
            cv2.polylines(frame, [self.exclude_polygon], True, (0, 0, 200), 2, cv2.LINE_AA)
            cv2.putText(frame, "EXCLUDED", 
                        (self.exclude_polygon[0][0] + 5, self.exclude_polygon[0][1] + 25),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 0, 200), 2)
        
        # --- Draw counting line ---
        cv2.line(frame, (self.line_x, self.line_y_start), 
                 (self.line_x, self.line_y_end), (0, 255, 255), 3, cv2.LINE_AA)
        cv2.putText(frame, "INSIDE", (self.line_x - 120, self.line_y_start - 10),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 255), 2)
        cv2.putText(frame, "OUTSIDE", (self.line_x + 10, self.line_y_start - 10),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 255), 2)
        
        # --- Draw detections and trails ---
        for det in detections:
            track_id = det['track_id']
            x1, y1, x2, y2 = det['bbox']
            cx, cy = det['centroid']
            
            if track_id in self.counted_ids:
                event = next((e for e in self.events if e['track_id'] == track_id), None)
                if event and event['direction'] == 'ENTER':
                    color = (0, 255, 0)   # Green = entered
                else:
                    color = (0, 0, 255)   # Red = exited
            else:
                color = (255, 200, 0)     # Cyan = tracked, not counted yet
            
            # Bounding box
            cv2.rectangle(frame, (int(x1), int(y1)), (int(x2), int(y2)), 
                         color, 2, cv2.LINE_AA)
            
            # Label with ID and area
            area = self._bbox_area(det['bbox'])
            label = f"ID:{track_id}"
            (tw, th), _ = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.6, 2)
            cv2.rectangle(frame, (int(x1), int(y1) - th - 10), 
                         (int(x1) + tw + 5, int(y1)), color, -1)
            cv2.putText(frame, label, (int(x1) + 2, int(y1) - 5),
                       cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 0, 0), 2)
            
            # Centroid dot
            cv2.circle(frame, (int(cx), int(cy)), 5, color, -1)
            
            # Trajectory trail
            history = list(self.track_history[track_id])
            trail = history[-TRAIL_LENGTH:]
            for i in range(1, len(trail)):
                alpha = i / len(trail)
                thickness = max(1, int(3 * alpha))
                pt1 = (int(trail[i-1][0]), int(trail[i-1][1]))
                pt2 = (int(trail[i][0]), int(trail[i][1]))
                cv2.line(frame, pt1, pt2, color, thickness, cv2.LINE_AA)
        
        # --- Stats overlay ---
        panel_h = 110
        overlay = frame[:panel_h, :].copy()
        cv2.rectangle(frame, (0, 0), (w, panel_h), (0, 0, 0), -1)
        cv2.addWeighted(overlay, 0.3, frame[:panel_h, :], 0.7, 0, frame[:panel_h, :])
        
        cv2.putText(frame, f"ENTERED: {self.enter_count}", (20, 40),
                   cv2.FONT_HERSHEY_SIMPLEX, 1.2, (0, 255, 0), 3, cv2.LINE_AA)
        cv2.putText(frame, f"EXITED: {self.exit_count}", (20, 80),
                   cv2.FONT_HERSHEY_SIMPLEX, 1.0, (0, 0, 255), 2, cv2.LINE_AA)
        cv2.putText(frame, f"Filtered: {filtered_count} (area/ROI)", (20, 105),
                   cv2.FONT_HERSHEY_SIMPLEX, 0.5, (150, 150, 150), 1, cv2.LINE_AA)
        
        cv2.putText(frame, f"Frame: {frame_no} | FPS: {fps:.1f}", (w - 350, 35),
                   cv2.FONT_HERSHEY_SIMPLEX, 0.7, (200, 200, 200), 2, cv2.LINE_AA)
        cv2.putText(frame, f"Active tracks: {len(detections)}", (w - 350, 65),
                   cv2.FONT_HERSHEY_SIMPLEX, 0.7, (200, 200, 200), 2, cv2.LINE_AA)
        cv2.putText(frame, "BoT-SORT (Kalman+Hungarian)", (w - 380, 95),
                   cv2.FONT_HERSHEY_SIMPLEX, 0.5, (150, 150, 150), 1, cv2.LINE_AA)
        
        return frame
    
    def process_video(self, video_path, output_path, show_preview=False):
        """Process video with enhanced filtering and polygon ROI."""
        cap = cv2.VideoCapture(video_path)
        if not cap.isOpened():
            raise FileNotFoundError(f"Cannot open video: {video_path}")
        
        w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
        h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
        fps = cap.get(cv2.CAP_PROP_FPS)
        total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        
        print(f"Video: {w}x{h} @ {fps:.1f} FPS, {total_frames} frames")
        print(f"Duration: {total_frames/fps:.1f}s")
        print(f"Counting line: x={self.line_x}, y=[{self.line_y_start}, {self.line_y_end}]")
        print(f"Min bbox area: {self.min_bbox_area} px²")
        print(f"Tracker: {self.tracker_config}")
        print("=" * 60)
        
        fourcc = cv2.VideoWriter_fourcc(*'mp4v')
        # Adjust output FPS to match the frame skip
        out = cv2.VideoWriter(output_path, fourcc, fps / self.frame_skip, (w, h))
        
        frame_no = 0
        processed_frames = 0
        start_time = time.time()
        total_filtered = 0
        
        while cap.isOpened():
            ret, frame = cap.read()
            if not ret:
                break
            
            frame_no += 1
            
            if frame_no % self.frame_skip != 0:
                continue
                
            processed_frames += 1
            timestamp = frame_no / fps
            
            # --- Run YOLOv8 with BoT-SORT tracking ---
            # Internally uses:
            #   1. Kalman Filter for motion prediction
            #   2. Hungarian Algorithm for detection-to-track assignment
            results = self.model.track(
                frame, persist=True, conf=self.confidence,
                classes=[self.person_class_id],
                tracker=self.tracker_config,
                verbose=False
            )
            
            detections = []
            frame_filtered = 0
            
            if results[0].boxes is not None and results[0].boxes.id is not None:
                boxes = results[0].boxes.xyxy.cpu().numpy()
                track_ids = results[0].boxes.id.cpu().numpy().astype(int)
                confidences = results[0].boxes.conf.cpu().numpy()
                
                for bbox, track_id, conf in zip(boxes, track_ids, confidences):
                    # --- Filter 1: Bounding box area ---
                    area = self._bbox_area(bbox)
                    if area < self.min_bbox_area:
                        self.filtered_by_area += 1
                        frame_filtered += 1
                        continue
                    
                    cx, cy = self._get_centroid(bbox)
                    
                    # Update tracking history (even if outside ROI)
                    self.track_history[track_id].append((cx, cy))
                    self.track_frames[track_id] += 1
                    
                    # Check for line crossing (ROI check is inside)
                    crossing = self._check_crossing(
                        track_id, cx, cy, frame_no, timestamp
                    )
                    
                    if crossing:
                        print(f"  [{timestamp:6.1f}s] Frame {frame_no}: "
                              f"Track #{track_id} → {crossing} "
                              f"(Total entered: {self.enter_count})")
                    
                    detections.append({
                        'track_id': track_id, 'bbox': bbox,
                        'centroid': (cx, cy), 'confidence': conf
                    })
            
            total_filtered += frame_filtered
            
            # Draw overlays
            elapsed = time.time() - start_time
            current_fps = frame_no / elapsed if elapsed > 0 else 0
            annotated = self._draw_overlay(
                frame, detections, total_filtered, frame_no, current_fps
            )
            out.write(annotated)
            
            if show_preview:
                preview = cv2.resize(annotated, (960, 540))
                cv2.imshow('People Counter V2', preview)
                if cv2.waitKey(1) & 0xFF == ord('q'):
                    break
            
            if frame_no % 200 == 0:
                pct = frame_no / total_frames * 100
                print(f"  Progress: {frame_no}/{total_frames} ({pct:.0f}%) "
                      f"| Enter: {self.enter_count} | Exit: {self.exit_count} "
                      f"| Filtered: {total_filtered} | FPS: {current_fps:.1f}")
        
        cap.release()
        out.release()
        if show_preview:
            cv2.destroyAllWindows()
        
        total_time = time.time() - start_time
        
        print("=" * 60)
        print(f"Processing complete!")
        print(f"  Total time: {total_time:.1f}s")
        print(f"  Avg FPS: {frame_no/total_time:.1f}")
        print(f"  Filtered by area: {self.filtered_by_area}")
        print(f"  Output saved to: {output_path}")
        
        return {
            'enter_count': self.enter_count,
            'exit_count': self.exit_count,
            'events': self.events,
            'total_frames': frame_no,
            'processing_time': total_time,
            'filtered_by_area': self.filtered_by_area
        }

## 4. Visualize Setup

Preview the polygon ROI, exclusion zone, and counting line on sample frames.

In [ ]:
def preview_setup_v2(video_path, line_x, line_y_start, line_y_end,
                     roi_polygon, exclude_polygon=None, frame_idx=0):
    """Preview polygon ROI, exclusion zone, and counting line."""
    cap = cv2.VideoCapture(video_path)
    cap.set(cv2.CAP_PROP_POS_FRAMES, frame_idx)
    ret, frame = cap.read()
    cap.release()
    
    if not ret:
        print("Cannot read frame")
        return
    
    h, w = frame.shape[:2]
    
    # Dim outside ROI
    mask = np.zeros((h, w), dtype=np.uint8)
    cv2.fillPoly(mask, [roi_polygon], 255)
    if exclude_polygon is not None and USE_EXCLUSION_ZONE:
        cv2.fillPoly(mask, [exclude_polygon], 0)
    frame[mask == 0] = (frame[mask == 0] * 0.4).astype(np.uint8)
    
    # Draw ROI polygon (green)
    cv2.polylines(frame, [roi_polygon], True, (0, 255, 0), 3, cv2.LINE_AA)
    cv2.putText(frame, "ROI Polygon", 
                (roi_polygon[0][0] + 5, roi_polygon[0][1] + 25),
                cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)
    
    # Draw exclusion zone (red)
    if exclude_polygon is not None and USE_EXCLUSION_ZONE:
        cv2.polylines(frame, [exclude_polygon], True, (0, 0, 255), 2, cv2.LINE_AA)
        cv2.putText(frame, "EXCLUDED (interior)",
                    (exclude_polygon[0][0] + 5, exclude_polygon[0][1] + 20),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 0, 255), 2)
    
    # Counting line (yellow)
    cv2.line(frame, (line_x, line_y_start), (line_x, line_y_end),
             (0, 255, 255), 3, cv2.LINE_AA)
    cv2.putText(frame, "INSIDE", (line_x - 120, line_y_start - 10),
               cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 255), 2)
    cv2.putText(frame, "OUTSIDE", (line_x + 10, line_y_start - 10),
               cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 255), 2)
    
    preview_path = "setup_preview_v2.jpg"
    cv2.imwrite(preview_path, frame)
    print(f"Preview saved to: {preview_path}")
    return frame

# Preview at different timestamps
for t_sec in [0, 30, 60]:
    _ = preview_setup_v2(
        VIDEO_INPUT, LINE_X, LINE_Y_START, LINE_Y_END,
        ROI_POLYGON, EXCLUDE_POLYGON if USE_EXCLUSION_ZONE else None,
        frame_idx=int(t_sec * 30)
    )

## 5. Run the Pipeline

In [ ]:
# Initialize V2 counter
counter = PeopleCounterV2(
    model_name=MODEL_NAME,
    confidence=CONFIDENCE_THRESHOLD,
    line_x=LINE_X,
    line_y_start=LINE_Y_START,
    line_y_end=LINE_Y_END,
    roi_polygon=ROI_POLYGON,
    crossing_margin=CROSSING_MARGIN,
    track_history_len=TRACK_HISTORY_LENGTH,
    min_track_hits=MIN_TRACK_HITS,
    min_bbox_area=MIN_BBOX_AREA,
    tracker_config=TRACKER_CONFIG,
    exclude_polygon=EXCLUDE_POLYGON if USE_EXCLUSION_ZONE else None,
    frame_skip=FRAME_SKIP
)

# Run processing
results = counter.process_video(
    video_path=VIDEO_INPUT,
    output_path=VIDEO_OUTPUT,
    show_preview=SHOW_PREVIEW
)

## 6. Results Summary

In [ ]:
print("=" * 60)
print("       PEOPLE COUNTING RESULTS (V2 — Enhanced)")
print("=" * 60)
print(f"")
print(f"  📥 People ENTERED:  {results['enter_count']}")
print(f"  📤 People EXITED:   {results['exit_count']}")
print(f"  ─────────────────────────────")
print(f"  📊 Total movements: {results['enter_count'] + results['exit_count']}")
print(f"  🚫 Filtered (small bbox): {results['filtered_by_area']}")
print(f"")
print(f"  🎬 Frames processed: {results['total_frames']}")
print(f"  ⏱️  Processing time: {results['processing_time']:.1f}s")
print(f"  📹 Output video:    {VIDEO_OUTPUT}")
print(f"")
print("=" * 60)

# Detailed event log
if results['events']:
    print(f"\n📋 Detailed Event Log:")
    print(f"{'Time':>8s}  {'Frame':>6s}  {'Track':>6s}  {'Direction'}")
    print(f"{'─'*8}  {'─'*6}  {'─'*6}  {'─'*9}")
    for event in results['events']:
        t = event['timestamp']
        arrow = '→ IN ' if event['direction'] == 'ENTER' else '← OUT'
        print(f"{t:7.1f}s  {event['frame']:6d}  #{event['track_id']:<5d}  {arrow}")

## 7. Method Description

### System Architecture

```
┌──────────────────────────────────────────────────────────────┐
│                    INPUT: Video Frame                        │
│                         ↓                                    │
│  ┌──────────────────────────────────────────┐               │
│  │  YOLOv8m — Person Detection              │               │
│  │  • Pre-trained on COCO (class 0: person)  │               │
│  │  • Confidence threshold: 0.4              │               │
│  │  • Output: bounding boxes + scores        │               │
│  └────────────────┬─────────────────────────┘               │
│                   ↓                                          │
│  ┌──────────────────────────────────────────┐               │
│  │  BoT-SORT — Multi-Object Tracking        │               │
│  │  ├─ Kalman Filter (motion prediction)     │               │
│  │  │  State: [cx,cy,ar,h,vx,vy,va,vh]       │               │
│  │  │  Predicts position during occlusion     │               │
│  │  │                                         │               │
│  │  ├─ Hungarian Algorithm (assignment)       │               │
│  │  │  Cost = IoU dist + appearance dist      │               │
│  │  │  Optimal detection↔track matching       │               │
│  │  │                                         │               │
│  │  └─ Output: boxes + persistent IDs         │               │
│  └────────────────┬─────────────────────────┘               │
│                   ↓                                          │
│  ┌──────────────────────────────────────────┐               │
│  │  Post-Processing & Filtering             │               │
│  │  ├─ Min bbox area filter (15,000 px²)     │               │
│  │  ├─ Polygon ROI check                     │               │
│  │  ├─ Centroid tracking (foot-level)        │               │
│  │  └─ Line-crossing + direction classify    │               │
│  └────────────────┬─────────────────────────┘               │
│                   ↓                                          │
│     ENTER count / EXIT count / Event Log                     │
└──────────────────────────────────────────────────────────────┘
```

### How Entry Direction is Classified

1. **Virtual Counting Line**: Vertical line at x=620 (door frame).
   Left = inside, Right = outside.

2. **Centroid Tracking**: Bottom-center of bbox (foot level) tracked
   across frames via BoT-SORT persistent IDs.

3. **Hysteresis Margin** (40px): Prevents flickering when person is
   exactly on the line.

4. **Crossing Detection**:
   - Right → Left = **ENTER** (outside → inside)
   - Left → Right = **EXIT** (inside → outside)

5. **Trajectory Confirmation**: Verified using last 10 centroid positions.

6. **Single-Count Guard**: Each track ID counted only once.

### Kalman Filter Role
- Maintains velocity estimates for each tracked person
- Predicts where the person should be in the next frame
- Bridges gaps when person is briefly occluded by door frame
- State prediction used by Hungarian algorithm for cost calculation

### Hungarian Algorithm Role
- Solves the linear assignment problem optimally
- Matches N detections to M existing tracks
- Minimizes total cost (IoU distance + appearance features)
- Ensures consistent ID assignment across frames

### Limitations
- Occlusion near the door can cause ID switches despite Kalman prediction
- People standing still on the counting line may not be counted
- Counting line position must be calibrated per camera
- Groups entering simultaneously may be partially missed due to overlap